# 직접 해보기 01 · 경보선을 어디에 그을까

미국 NOAA 우주환경예보센터의 ‘다음 날 플레어 확률’과 실제 기록을 짝지은 표(Camporeale 2025, Zenodo 10.5281/zenodo.16620803, CC BY 4.0)를 읽는다.

교육용 재계산이다. 원 논문의 모든 점수를 재현한 것이 아니며, 실시간 예보나 설비 운영의 근거로 쓸 수 없다. 같은 계산이 `analysis.py`에도 들어 있다.

In [ ]:
import analysis as A
import matplotlib.pyplot as plt

rows = A.load()
print("파일 지문", A.sha8(), "· 행", len(rows))
X = A.select(rows, "X", 1, "all")
print("X급 · 다음 날 · 1998–2024:", len(X), "일")

## 1. 원 보고서부터
`data/20240510RSGA.txt`의 `III. Event probabilities` 대목에서 X급 75/75/75의 대상 날짜를 확인한다. 표에서는 발행일 2024-05-10의 세 행이다.

In [ ]:
for r in rows:
    if r[0] == "2024-05-10" and r[3] == "X":
        print(r)

## 2. 교정 곡선
예보한 확률을 10%p씩 묶어, 묶음의 평균 확률과 실제 발생 비율을 비교한다. 점선 위면 말한 대로다.

In [ ]:
bins = A.reliability(X, 10)
fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.scatter([b["mean_p"] for b in bins], [b["rate"] for b in bins], s=[b["n"] ** 0.5 * 4 for b in bins])
ax.set_xlabel("예보 확률(묶음 평균)"); ax.set_ylabel("실제 발생 비율"); plt.show()
for b in bins: print(b["lo"], b["n"], round(b["mean_p"], 3), round(b["rate"], 3))

## 3. 정답률의 함정

In [ ]:
events = sum(r[5] for r in X)
print(events, "일 발생 · 늘 ‘없음’이라고 하면 정답률", round((len(X) - events) / len(X) * 100, 1), "%")

## 4. 경보선을 50%에 긋는다
경보는 ‘확률 ≥ 경보선’일 때 울린다.

In [ ]:
print(A.four_boxes(X, 0.5))

## 5. 경보선을 10%로 내린다

In [ ]:
print(A.four_boxes(X, 0.1))
ths = [t / 100 for t in range(1, 100)]
fig, ax = plt.subplots(figsize=(6, 3.6))
for key, label in (("false_alarm", "괜히 울린 날"), ("miss", "놓친 날"), ("hit", "맞힌 날")):
    pts = [(t, A.four_boxes(X, t)[key]) for t in ths]
    pts = [(t, v) for t, v in pts if v > 0]
    ax.plot([p[0] for p in pts], [p[1] for p in pts], label=label)
ax.set_yscale("log"); ax.set_xlabel("경보선"); ax.set_ylabel("날짜 수"); ax.legend(); plt.show()

## 6. 공짜 예보와 비교한다
‘어제 X급이 있었으면 오늘도 있다’는 지속 예보. 같은 표의 달력 날짜로 어제를 찾는다(날짜가 빠진 곳은 뺀다).

In [ ]:
print(A.persistence(rows, "X", "all"))

## 7. 조건을 바꿔 본다
기간을 2020–2024로 좁히면? 그 차이를 곧바로 예보 기술의 발전으로 단정하지 말고, 자료·조건·결과를 함께 적어 두자.

In [ ]:
for period in ("all", "recent"):
    sel = A.select(rows, "X", 1, period)
    print(period, len(sel), A.four_boxes(sel, 0.5), A.four_boxes(sel, 0.1))